In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

%pip install kagglehub catboost lightgbm tqdm -q

In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing_values = df.isnull().sum()

print("Missing Values per Column:")
print(missing_values)

In [ ]:
df = df.dropna(subset=['P_2', 'B_2', 'D_43','D_42', 'D_137', 'D_138','D_142','D_143','D_144', 'D_145'])
missing_values = df.isnull().sum()

print("Missing Values after Dropping Rows:")
print(missing_values)

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print("Number of Duplicate Samples: ")
print(duplicates)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

categorical_cols

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
import numpy as np

X = df.drop("Target", axis=1)
y = df["Target"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
f1_scores = []

model = CatBoostClassifier(
    iterations=100,
    depth=8,
    learning_rate=0.1,
    verbose=0,
    random_state=42
)

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc_scores.append(accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))

print("Model Evaluation (Stratified K-Fold)")
print("-" * 40)
print(f"Average Accuracy : {np.mean(acc_scores):.4f}")
print(f"Average F1 Score : {np.mean(f1_scores):.4f}")
print("-" * 40)

In [ ]:
# Task 1: Write your code here:
importances = model.feature_importances_
features = X.columns

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.sort_values(
    by="Importance", ascending=False
).iloc[0]["Feature"]

print(" Golden Feature:", golden_feature)

In [ ]:
# Task Bonus: Write your code here: